# 07 — Create Subscription Orders (inactive)

The final creation step for INACTIVE subscriptions (`SubscriptionEndDate`
already in the past) — no Voyager lookup, no new address; each order's
`shipAddId` is the account's existing default service address, attached in
`06_Attach_Inactive_Addresses.ipynb`.

`build_subscription_order_payload`, `create_order_for_subscription`, and
`create_all_orders` are shared with the active pipeline — they live in
`onebill_common.py`. This notebook just loads the inactive-pipeline data
and calls them.

**vBill -> OneBill field mapping** — identical to the active pipeline
(see `07_Create_Subscription_Orders.ipynb`), with one difference:

| OneBill field | Source |
|---|---|
| Radius Username | vBill `SubscriptionLabel`, directly — no Voyager circuits lookup (set in `05_Fetch_Inactive_Subscriptions.ipynb`) |


## 1. Setup — load everything the previous notebooks produced


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_inactive_subscription_orders")

df_subscriptions   = try_load_df("subscriptions_inactive_resolved", dtype=SUBSCRIPTIONS_RESOLVED_DTYPES)
df_address_results = try_load_df("address_results_inactive", dtype={"ship_add_id": str})
df_plan_mapping     = try_load_df("plan_mapping", dtype=str)
df_contacts          = try_load_df("contacts")

if df_subscriptions is None or df_subscriptions.empty:
    logger.warning("No inactive subscriptions found — run 05_Fetch_Inactive_Subscriptions.ipynb first, or there simply were none this run.")
    df_subscriptions = pd.DataFrame()
if df_address_results is None:
    logger.warning("06_Attach_Inactive_Addresses.ipynb hasn't been run yet (or produced no rows) — no inactive subscriptions are ready for order creation.")
    df_address_results = pd.DataFrame(columns=["SubscriptionUSN", "status", "ship_add_id", "error"])

if df_plan_mapping is None:
    logger.warning("03_plan_code_mapping.csv not found — every subscription falls back to STATIC_FALLBACK_PLAN.")
    df_plan_mapping = pd.DataFrame(columns=["PlanCode", "product_name", "priceplan_name"])

if df_contacts is None and ATTACH_CONTACT_SUMMARY_TO_ORDER:
    logger.warning("01_contacts_by_account.csv not found — orders will be created without a contact summary.")

logger.info(
    f"{len(df_subscriptions):,} inactive subscriptions, {len(df_address_results):,} address results, "
    f"{len(df_plan_mapping):,} plan mappings, "
    f"{len(df_contacts) if df_contacts is not None else 0:,} contact summaries"
)


## 2. Join subscriptions with their (reused) default address


In [ ]:
if df_subscriptions.empty:
    ready = df_subscriptions.copy()
    not_ready = df_subscriptions.copy()
else:
    df_work = df_subscriptions.merge(
        df_address_results[["SubscriptionUSN", "status", "ship_add_id", "error"]].rename(
            columns={"status": "AddressStatus", "error": "AddressError"}
        ),
        on="SubscriptionUSN", how="left",
    )
    ready = df_work[df_work["AddressStatus"] == "reused_default"]
    not_ready = df_work[df_work["AddressStatus"] != "reused_default"]
    logger.info(f"{len(ready):,} inactive subscriptions have a reused default address and are ready for order creation; "
                f"{len(not_ready):,} do not and will be skipped")

not_ready[["SubscriptionUSN", "AddressStatus", "AddressError"]].head(20) if not not_ready.empty else not_ready


## 3. Plan resolution


In [ ]:
resolve_plan = make_plan_resolver(df_plan_mapping)


## 4. Contact-summary lookup (optional)

Joins each subscription's *original* vBill AccountCode to its primary
contact from `01_Fetch_Contacts.ipynb` — see `ATTACH_CONTACT_SUMMARY_TO_ORDER`.


In [ ]:
contact_summary_attributes = make_contact_summary_fn(df_contacts)


## 5. Run (parallel driver)


In [ ]:
if ready.empty:
    order_results_df = pd.DataFrame(columns=[
        "SubscriptionUSN", "TargetAccountNumber", "status", "plan_matched",
        "productName", "priceplanName", "onebill_order_id", "error",
    ])
else:
    order_results_df = create_all_orders(ready, resolve_plan, contact_summary_attributes)

order_results_df.head(20)


## 6. Save


In [ ]:
save_df("order_results_inactive", order_results_df)
